# Baseline Risk Model

This notebook establishes the conventional baseline: allowed alternative-financial features are used to predict `financial_stress_event`. It does not implement calibration correction, confidence estimation, OOD detection, evidence-quality scoring, or a Safety Gate.

In [ ]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src.models.baseline import (
    BASELINE_FEATURES,
    SELECTED_THRESHOLD,
    make_train_validation_test_split,
    run_baseline_experiment,
)

sns.set_theme(style='whitegrid')

## Dataset Loading

The baseline uses `data/processed/synthetic/model_dataset.csv`. The target is `financial_stress_event`. The generated probability `financial_stress_probability` is not used as an input feature.

In [ ]:
dataset_path = PROJECT_ROOT / 'data' / 'processed' / 'synthetic' / 'model_dataset.csv'
dataset = pd.read_csv(dataset_path)
dataset.shape, dataset['financial_stress_event'].mean()

## Feature Selection

Only the 13 allowed alternative-financial features are used. The model does not receive `applicant_id`, population labels, latent variables, or `financial_stress_probability`.

In [ ]:
list(BASELINE_FEATURES)

## Train / Validation / Test Split

The split is 70% train, 15% validation, and 15% test. Both split steps are stratified on `financial_stress_event`. The validation set is used for threshold sensitivity; the test set is held out until final evaluation.

In [ ]:
splits = make_train_validation_test_split(dataset, seed=42)
split_summary = pd.DataFrame(
    {
        name: {
            'rows': len(split),
            'event_rate': split['financial_stress_event'].mean(),
        }
        for name, split in splits.items()
    }
).T
split_summary

## Logistic Regression and HistGradientBoosting

The experiment trains logistic regression and `HistGradientBoostingClassifier`. Both produce `P(financial_stress_event = 1)`.

In [ ]:
metrics = run_baseline_experiment(
    dataset_path=dataset_path,
    output_dir=PROJECT_ROOT / 'experiments' / 'baseline',
    seed=42,
)
comparison = pd.read_csv(PROJECT_ROOT / 'experiments' / 'baseline' / 'model_comparison.csv')
comparison

## Threshold Policy

The documented initial threshold is 0.50. Nearby thresholds are evaluated on the validation set as sensitivity analysis. Lower thresholds increase recall and positive prediction rate; higher thresholds increase selectivity but miss more stress events.

In [ ]:
sensitivity_rows = []
for model_name, model_metrics in metrics['models'].items():
    for row in model_metrics['validation_threshold_sensitivity']:
        sensitivity_rows.append({'model': model_name, **row})
pd.DataFrame(sensitivity_rows)

## Evaluation

The held-out test set is evaluated with ranking, classification, and probability-quality metrics.

In [ ]:
test_metric_rows = []
for model_name, model_metrics in metrics['models'].items():
    row = {'model': model_name}
    row.update(model_metrics['test_metrics'])
    test_metric_rows.append(row)
pd.DataFrame(test_metric_rows).drop(columns=['confusion_matrix'])

In [ ]:
{model_name: model_metrics['test_metrics']['confusion_matrix'] for model_name, model_metrics in metrics['models'].items()}

## Calibration Diagnostics

These plots diagnose calibration only. No calibration correction is applied in this phase.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
ax.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Perfect calibration')
for model_name, model_metrics in metrics['models'].items():
    calibration = pd.DataFrame(model_metrics['calibration_diagnostics'])
    ax.plot(
        calibration['mean_predicted_probability'],
        calibration['observed_event_rate'],
        marker='o',
        label=model_name,
    )
ax.set_xlabel('Mean predicted probability')
ax.set_ylabel('Observed event rate')
ax.set_title('Reliability Diagram')
ax.legend()
plt.tight_layout()

In [ ]:
logistic_predictions = pd.read_csv(PROJECT_ROOT / 'experiments' / 'baseline' / 'predictions_logistic.csv')
tree_predictions = pd.read_csv(PROJECT_ROOT / 'experiments' / 'baseline' / 'predictions_tree.csv')
plot_predictions = pd.concat(
    [
        logistic_predictions.assign(model='logistic_regression'),
        tree_predictions.assign(model='hist_gradient_boosting'),
    ],
    ignore_index=True,
)

sns.histplot(data=plot_predictions, x='predicted_probability', hue='model', bins=40, element='step', stat='density', common_norm=False)
plt.axvline(SELECTED_THRESHOLD, color='black', linestyle='--')
plt.title('Test-Set Predicted Probability Distribution')
plt.tight_layout()

## Interpretation and Limitations

The baseline is a predictive benchmark, not the proposed research contribution. Its role is to establish conventional automated decision behavior before evidence-aware and uncertainty-aware safety gating is introduced.

Current results are moderate rather than near-perfect, which reduces immediate leakage concern. However, the target is synthetic and generated from the same feature family used for prediction, so real-world validity cannot be inferred. Calibration is diagnostic only here; correction belongs in a later phase.